# Order Fulfillment> **Layer:** Gold &nbsp;|&nbsp; **Process:** `gold/order-fulfillment` &nbsp;|&nbsp; **Domains:** Sales, Finance &nbsp;|&nbsp; **Owner:** Sales Analytics Team &nbsp;|&nbsp; **Updated:** Daily---## 1. Business Process OverviewThe Order Fulfillment process captures every product sold to a customer through a sales order. It is the authoritative record of commercial transactions and forms the foundation for revenue reporting, product performance analysis, and customer purchasing behaviour.**Grain:** One row per sales order line item — identified by the combination of Sales Order Number and Sales Order Line ID.**Business Questions Answered:**- What were the total sales for each month last year?- Which products generated the most revenue in the last quarter?- How many sales orders were placed by each customer in a given period?- What is the trend of sales volume over time?- What is the average order value?**Source System(s):**- [AdventureWorks ERP](../../source-system/adventureworks-erp.md)---## 2. Conceptual Model> A high-level view of the business entities involved in this process. Click any entity to navigate to its master data page. For full attribute definitions see [Section 4](#4-attributes--history-tracking).```mermaidflowchart TD    CUSTOMER["👤 Customer"]    SALES_ORDER["📋 Sales Order"]    ORDER_LINE["📝 Order Line"]    PRODUCT["📦 Product"]    DATE["📅 Date"]    ADDRESS["🏠 Address"]    CUSTOMER -->|"places"| SALES_ORDER    SALES_ORDER -->|"contains"| ORDER_LINE    ORDER_LINE -->|"references"| PRODUCT    ORDER_LINE -->|"placed on"| DATE    CUSTOMER -->|"ships to"| ADDRESS    click CUSTOMER "../../master/customer.md" "Go to: Customer master data"    click PRODUCT "../../master/product.md" "Go to: Product master data"    click ADDRESS "../../master/address.md" "Go to: Address master data"    click DATE "../../master/date.md" "Go to: Date master data"```---## 3. Data Quality Rules| Rule | Threshold | Severity ||------|-----------|----------|| Line total must be positive | 100% | Critical || Order date must not be null | 100% | Critical || Unit price must be non-negative | 100% | Critical || Order quantity must be positive | 100% | Critical || Customer must match a known customer | 100% | Critical || Product must match a known product | 100% | Critical |---## 4. Attributes & History Tracking> ### Attribute Type Legend>> | Code | Business Name | What it means |> |------|---------------|---------------|> | **ID** | Identifier | The unique label that identifies this record — like an order number or customer ID. Never changes. |> | **REF** | Reference | A pointer to a related business entity, such as the customer or product on an order. |> | **CURRENT** | Current Value | We keep the latest value only. If it changes, the old value is not retained. (e.g. a customer's phone number) |> | **HISTORY** | Full History | Every change is recorded over time so you can see what the value was at any point in the past. |> | **STATIC** | Fixed Value | This value never changes by nature (e.g. the name of a month). |> | **MEASURE** | Measured Value | A number that can be added up, averaged, or otherwise calculated across records. |---### Transactional Data: Sales Orders*The core record of what happened — one row for every line on every sales order.*| Business Name | Description | Type | Aggregation ||---------------|-------------|------|-------------|| Sales Order Number | Unique identifier for the sales order header | ID | — || Sales Order Line ID | Unique identifier for the line item within an order | ID | — || Order Date | Date the sales order was placed | REF → [Date](../../master/date.md) | — || Order Quantity | Number of units of a product ordered | MEASURE | SUM || Unit Price | Price per unit charged to the customer at time of order | MEASURE | AVERAGE || Unit Price Discount | Discount applied to the unit price | MEASURE | SUM || Line Total | Total revenue for this line item (quantity × discounted price) | MEASURE | SUM |---### Master Data: [Customer](../../master/customer.md)*Who placed the order. Customer details reflect the most current known information.*| Business Name | Description | Type ||---------------|-------------|------|| Customer ID | Unique identifier for the customer | ID || First Name | First name of the customer | CURRENT || Last Name | Last name of the customer | CURRENT || Company Name | Company name if the customer is a business | CURRENT || Email Address | Email address of the customer | CURRENT || Phone | Phone number of the customer | CURRENT |---### Master Data: [Product](../../master/product.md)*What was sold. Product details reflect the most current known information.*| Business Name | Description | Type ||---------------|-------------|------|| Product ID | Unique identifier for the product | ID || Product Name | Name of the product | CURRENT || Product Number | Unique product number | CURRENT || Color | Color variant of the product | CURRENT || Size | Size variant of the product | CURRENT || List Price | Standard list price of the product | CURRENT |---### Master Data: [Address](../../master/address.md)*Where the order was shipped. Address details reflect the most current known information.*| Business Name | Description | Type ||---------------|-------------|------|| Address ID | Unique identifier for the address | ID || Address Line 1 | First line of the street address | CURRENT || Address Line 2 | Second line of the street address | CURRENT || City | City of the address | CURRENT || State / Province | State or province | CURRENT || Country / Region | Country or region | CURRENT || Postal Code | Postal code | CURRENT |---### Master Data: [Date](../../master/date.md)*The calendar reference. All date attributes are fixed by nature and never change.*| Business Name | Description | Type ||---------------|-------------|------|| Date Key | Unique key for the date | ID || Full Date | The complete calendar date | STATIC || Day of Week | Day of the week (1–7) | STATIC || Day of Month | Day of the month (1–31) | STATIC || Day of Year | Day of the year (1–366) | STATIC || Week of Year | Week number of the year | STATIC || Month | Month number (1–12) | STATIC || Month Name | Name of the month | STATIC || Quarter | Quarter of the year (1–4) | STATIC || Year | Calendar year | STATIC |---## 5. Functional Lineage> Data flows upward from source systems through Bronze (raw), Silver (cleansed), and Gold (modelled) layers. Click any node to navigate to that asset's documentation page.```mermaidflowchart BT    subgraph SOURCE["Source Systems"]        SRC["🗄️ AdventureWorks ERP"]    end    subgraph BRONZE["Bronze — Raw"]        B1["📄 raw-sales-orders"]        B2["📄 raw-customers"]        B3["📄 raw-products"]    end    subgraph SILVER["Silver — Cleansed"]        S1["📄 cleansed-sales-orders"]        S2["📄 cleansed-customers"]        S3["📄 cleansed-products"]    end    subgraph GOLD["Gold — Modelled"]        G1["⭐ order-fulfillment\n(this page)"]    end    SRC -->|"extract"| B1    SRC -->|"extract"| B2    SRC -->|"extract"| B3    B1 -->|"cleanse\nnull handling, dedup, type cast"| S1    B2 -->|"cleanse\nnull handling, standardise"| S2    B3 -->|"cleanse\nnull handling, standardise"| S3    S1 -->|"model\njoin, compute line total"| G1    S2 -->|"join"| G1    S3 -->|"join"| G1    click SRC "../../source-system/adventureworks-erp.md" "Go to: source-system/adventureworks-erp"    click B1 "../../bronze/raw-sales-orders.md" "Go to: bronze/raw-sales-orders"    click B2 "../../bronze/raw-customers.md" "Go to: bronze/raw-customers"    click B3 "../../bronze/raw-products.md" "Go to: bronze/raw-products"    click S1 "../../silver/cleansed-sales-orders.md" "Go to: silver/cleansed-sales-orders"    click S2 "../../silver/cleansed-customers.md" "Go to: silver/cleansed-customers"    click S3 "../../silver/cleansed-products.md" "Go to: silver/cleansed-products"```### Lineage Summary| From | Asset | To | Asset | Transformation | Description ||------|-------|----|-------|----------------|-------------|| Source System | [AdventureWorks ERP](../../source-system/adventureworks-erp.md) | Bronze | [raw-sales-orders](../../bronze/raw-sales-orders.md) | Extract | Full load of raw order data || Source System | [AdventureWorks ERP](../../source-system/adventureworks-erp.md) | Bronze | [raw-customers](../../bronze/raw-customers.md) | Extract | Full load of raw customer data || Source System | [AdventureWorks ERP](../../source-system/adventureworks-erp.md) | Bronze | [raw-products](../../bronze/raw-products.md) | Extract | Full load of raw product data || Bronze | [raw-sales-orders](../../bronze/raw-sales-orders.md) | Silver | [cleansed-sales-orders](../../silver/cleansed-sales-orders.md) | Cleanse | Null handling, type casting, deduplication || Bronze | [raw-customers](../../bronze/raw-customers.md) | Silver | [cleansed-customers](../../silver/cleansed-customers.md) | Cleanse | Null handling, name standardisation || Bronze | [raw-products](../../bronze/raw-products.md) | Silver | [cleansed-products](../../silver/cleansed-products.md) | Cleanse | Null handling, price validation || Silver | [cleansed-sales-orders](../../silver/cleansed-sales-orders.md) | Gold | order-fulfillment *(this page)* | Model | Join to all master data, compute line total || Silver | [cleansed-customers](../../silver/cleansed-customers.md) | Gold | order-fulfillment *(this page)* | Join | Enrich orders with customer attributes || Silver | [cleansed-products](../../silver/cleansed-products.md) | Gold | order-fulfillment *(this page)* | Join | Enrich orders with product attributes |---## 6. Related Pages| Asset | Layer | Link ||-------|-------|------|| AdventureWorks ERP | Source System | [source-system/adventureworks-erp](../../source-system/adventureworks-erp.md) || Raw Sales Orders | Bronze | [bronze/raw-sales-orders](../../bronze/raw-sales-orders.md) || Raw Customers | Bronze | [bronze/raw-customers](../../bronze/raw-customers.md) || Raw Products | Bronze | [bronze/raw-products](../../bronze/raw-products.md) || Cleansed Sales Orders | Silver | [silver/cleansed-sales-orders](../../silver/cleansed-sales-orders.md) || Cleansed Customers | Silver | [silver/cleansed-customers](../../silver/cleansed-customers.md) || Cleansed Products | Silver | [silver/cleansed-products](../../silver/cleansed-products.md) || Customer | Master Data | [master/customer](../../master/customer.md) || Product | Master Data | [master/product](../../master/product.md) || Address | Master Data | [master/address](../../master/address.md) || Date | Master Data | [master/date](../../master/date.md) |---*Page ID: `gold/order-fulfillment` &nbsp;|&nbsp; Last updated: 2026-03-01 &nbsp;|&nbsp; Owner: Sales Analytics Team*

In [ ]:
-- Generated from specification-- Joining source tables based on common keysSELECT     t1.id,    t1.name,    t2.value,    t2.categoryFROM source_table_1 t1INNER JOIN source_table_2 t2     ON t1.id = t2.source_idWHERE t1.active = trueORDER BY t1.name;